In [1]:
import cv2
from tqdm import tqdm
import numpy as np
import glob
import tensorflow as tf
from tensorflow.keras import layers
import gc

In [2]:
import tensorflow as tf
from tensorflow.keras import layers

In [3]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [4]:
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 150

TRAIN_PATH = r'F:\DataSets\Dataset BUSI _1\train'
TEST_PATH = r'F:\DataSets\Dataset BUSI _1\test'

classes = {0 : "benign",
           1 : "malignant",
           2 : "normal"}

In [5]:
datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale = 1./255,
                                                         horizontal_flip = True,
                                                         vertical_flip = True,
                                                         rotation_range=5)

datagen_val = tf.keras.preprocessing.image.ImageDataGenerator(rescale = 1./255)


train_gen = datagen.flow_from_directory(color_mode='rgb',
                                        directory = TRAIN_PATH,
                                        batch_size = BATCH_SIZE,
                                        shuffle = True,
                                        target_size = (IMAGE_SIZE, IMAGE_SIZE))

valid_gen = datagen_val.flow_from_directory(color_mode='rgb',
                                        directory = TEST_PATH,
                                        batch_size = BATCH_SIZE,
                                        shuffle = False,
                                        target_size = (IMAGE_SIZE, IMAGE_SIZE))

Found 630 images belonging to 3 classes.
Found 150 images belonging to 3 classes.


In [6]:
def del_var(var):
    import gc
    try:
        del var
    except:
        pass
    gc.collect()

In [7]:
def create_pre_trained(base_model, pretrained_process_input):
    base_model.trainable = True
    model = tf.keras.models.Sequential()
    model.add(tf.keras.layers.Input(shape=(w, h, 3)))
    model.add(Lambda(pretrained_process_input, name='preprocessing', input_shape=(w, h, 3)))
    model.add(base_model)
    model.add(layers.Flatten())
    model.add(layers.Dense(32, activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.Dense(16, activation='relu'))
    model.add(layers.Dense(3, activation='softmax'))
    model.compile(loss='categorical_crossentropy',
            optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
                  metrics=['acc'])
    
    model.summary()
    return model

In [8]:
def plot_history(hist, history_param='acc'):
    import matplotlib.pyplot as plt
    plt.plot(hist.history[history_param])
    plt.plot(hist.history['val_'+history_param])
    plt.title('model '+ history_param)
    plt.ylabel(history_param)
    plt.xlabel('epoch')
    plt.legend(['train', 'test'], loc='upper left')
    plt.show()

In [9]:
w, h = 224, 224

# Mobile Net Trial

In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_preprocess_input
from tensorflow.keras.layers import Lambda
mob_net = create_pre_trained(MobileNetV2(weights="imagenet", include_top = False), mob_preprocess_input)

mob_net_history = mob_net.fit(train_gen, batch_size = BATCH_SIZE, epochs=EPOCHS,
                 validation_data = valid_gen ,callbacks = [CustomCallback()])

In [ ]:
plot_history(mob_net_history, 'acc')

In [ ]:
plot_history(mob_net_history, 'loss')

In [ ]:
mob_net.save('mob_net.h5')

In [ ]:
del_var(mob_net)
del_var(MobileNetV2)

# InceptionV3 Trial done

In [ ]:
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess_input
from tensorflow.keras.layers import Lambda
inception = create_pre_trained(InceptionV3(weights="imagenet", include_top = False), inception_preprocess_input)

inception_history = inception.fit(train_gen, batch_size = BATCH_SIZE, epochs=EPOCHS,
                 validation_data = valid_gen)

In [ ]:
plot_history(inception_history, 'acc')

In [ ]:
plot_history(inception_history, 'loss')

In [ ]:
inception.save('inception.h5')

In [ ]:
del_var(inception)
del_var(InceptionV3)

# Xception Trail

In [13]:
from tensorflow.keras.applications.resnet50 import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as ResNet50_preprocess_input
from tensorflow.keras.layers import Lambda

resnet50 = create_pre_trained(ResNet50(weights="imagenet", include_top = False), ResNet50_preprocess_input)

resnet50_history = ResNet50.fit(train_gen, batch_size = BATCH_SIZE, epochs=EPOCHS,
                 validation_data = valid_gen)

 4038656/94765736 [>.............................] - ETA: 22:19

KeyboardInterrupt: 

In [ ]:
plot_history(xception_history, 'acc')

In [ ]:
plot_history(xception_history, 'loss')

In [ ]:
xception.save('xception.h5')

In [ ]:
del_var(xception)
del_var(Xception)

In [12]:
gc.collect()

0

# PLots

In [ ]:
import matplotlib.pyplot as plt
history_param = 'acc'
plt.plot(mob_net_history.history['val_'+ history_param])

plt.plot(inception_history.history['val_'+ history_param])

plt.plot(efficientnet_history.history['val_'+ history_param])

plt.title('models val '+ history_param)
plt.ylabel(history_param)
plt.xlabel('epoch')
plt.legend(['MobileNetV2', 'InceptionV3' , 'EfficientNetB0'], loc='lower right')
plt.show()


In [ ]:
history_param = 'loss'
plt.plot(mob_net_history.history[history_param])

plt.plot(inception_history.history[history_param])

plt.plot(efficientnet_history.history[history_param])

plt.title('models '+ history_param)
plt.ylabel(history_param)
plt.xlabel('epoch')
plt.legend(['MobileNetV2', 'InceptionV3' ,'EfficientNetB0'], loc='upper right')
plt.show()


In [ ]:
# from sklearn import metrics
# from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report

# predicted = model.predict_generator(valid_gen)

# confusion_matrix = metrics.confusion_matrix(valid_gen.classes, np.argmax(predicted,axis=1))
# cm_display = metrics.ConfusionMatrixDisplay(confusion_matrix = confusion_matrix, display_labels = [0, 1, 2])
# cm_display.plot()
# plt.show()

# accuracy = accuracy_score(valid_gen.classes, np.argmax(predicted,axis=1))
# print("Accuracy   :", accuracy)